# 02 - Xu ly du lieu (Preprocessing)

Notebook nay trinh bay va giai thich pipeline lam sach + tien xu ly du lieu, dung chung cho huan luyen (`03_train.ipynb`) va cho AI Service khi phuc vu du doan. Toan bo logic nam trong `ai-models/src/data_utils.py`.

## 2.1. Kiem tra ban dau

- Kieu du lieu: `price`, `year`, `driven kms` dang o dang chuoi/so lan (vi du gia dang `'2 Ty 700 Trieu'`).
- Kiem tra trung lap: dung `url` cua tin dang lam khoa duy nhat.
- Kiem tra gia tri vo ly: nam san xuat ngoai khoang hop ly (loi crawl lam lech cot), so cua/so cho am hoac qua lon.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
from data_utils import load_raw, parse_price_to_million_vnd, split_brand_model
raw = load_raw('../data/data.csv')
print(raw.dtypes)
print('So dong trung theo url:', raw.duplicated(subset=['url']).sum())
print('Vi du parse gia:')
for s in ['2 Ty 700 Trieu', '666 Trieu', '365 Trieu']:
    print(' ', s, '->', parse_price_to_million_vnd(s))

## 2.2. Lam sach

**Vi sao lam theo cach nay:**
- Cot `price` la chuoi tieng Viet (`Ty`, `Trieu`) -> phai parse bang regex ve so trieu VND thay vi ep kieu truc tiep.
- Dung `dropna` cho cac truong quan trong thay vi dien gia tri, vi ty le thieu thap (xem Hinh 3, notebook EDA) nen khong dang danh doi do chinh xac.
- Loc ngoai lai theo nguong vat ly hop ly (km, so cua, so cho, khoang gia) thay vi cat theo phan vi, de giu duoc cang nhieu du lieu that cang tot ma van loai duoc loi crawl ro rang.

In [ ]:
from data_utils import clean_dataframe
df = clean_dataframe(raw)
print('Truoc lam sach:', raw.shape, '| Sau lam sach:', df.shape)
df.head()

## 2.3. Tao / chon dac trung

- Tach `car_name` thanh `brand` (hang xe) va `model` (dong xe) bang cach cat theo tu dau tien: mang them thong tin ve phan khuc/hang, giup model phan biet gia theo thuong hieu (xem Hinh 5).
- Khong tao them dac trung tong hop (vi du 'tuoi xe = nam hien tai - year') vi `year` da la dac trung tuyen tinh tot va cac model phi tuyen (Random Forest, Decision Tree) tu hoc duoc quan he nay.

## 2.4. Ma hoa & 2.5. Chuan hoa

| Nhom bien | Xu ly | Vi sao |
|---|---|---|
| So: `year`, `driven_kms`, `num_of_door`, `num_of_seat` | `StandardScaler` | SVR (khoang cach/kernel) rat nhay voi thang do; Linear Regression on dinh he so hon khi chuan hoa |
| Phan loai: `brand`, `model`, `series`, `assemble_place`, `engine_type`, `transmission` | `OneHotEncoder(handle_unknown='ignore')` | Khong co thu tu tu nhien giua cac gia tri; `handle_unknown='ignore'` giup model khong loi khi gap hang/dong xe moi luc du doan |

Decision Tree / Random Forest khong bat buoc phai chuan hoa bien so (cay quyet dinh khong nhay cam thang do), nhung de dung **chung mot `ColumnTransformer`** cho ca 4 model giup so sanh cong bang va don gian hoa code.

## 2.6. Chia du lieu

`train_test_split(test_size=0.2, random_state=42)` - 80% train / 20% test, co dinh `random_state` de tai lap ket qua. Day la bai toan hoi quy nen khong dung `stratify`.

**Tranh data leakage:** `ColumnTransformer` (StandardScaler/OneHotEncoder) duoc dat *ben trong* `sklearn.Pipeline`, chi `fit` tren `X_train` khi goi `pipeline.fit(X_train, y_train)`; khi danh gia tren `X_test`, pipeline chi `transform` (khong `fit` lai). Xem chi tiet trong `03_train.ipynb`.

## 2.7. Dau ra cua buoc nay

- Pipeline tien xu ly (nam trong `model.joblib` cung voi model, xem `03_train.ipynb`).
- `schema.json` (sinh ra o buoc train) mo ta ten cot, kieu, khoang gia tri hop le, danh sach nhan cua tung bien phan loai - duoc Backend dung de validate va Frontend dung de dung form nhap lieu.

### Toan bo noi dung `data_utils.py` (tham khao)

Dung chung giua EDA, huan luyen va AI Service.

In [ ]:
# -*- coding: utf-8 -*-
"""
data_utils.py
Ham dung chung de doc va lam sach du lieu xe cu tu bonbanh.com (Kaggle: danh911/gi-xe).
Dung lai o ca notebook EDA, notebook/train.py huan luyen, va AI Service khi can kiem tra dau vao.
"""
import re
import numpy as np
import pandas as pd

RAW_COLUMNS = [
    "car_name", "year", "price", "assemble_place", "series",
    "driven kms", "num_of_door", "num_of_seat", "engine_type",
    "transmission", "url",
]

FEATURE_COLUMNS = [
    "brand", "model", "series", "year", "driven_kms",
    "assemble_place", "engine_type", "transmission",
    "num_of_door", "num_of_seat",
]
TARGET_COLUMN = "price"
NUMERIC_FEATURES = ["year", "driven_kms", "num_of_door", "num_of_seat"]
CATEGORICAL_FEATURES = ["brand", "model", "series", "assemble_place", "engine_type", "transmission"]


def parse_price_to_million_vnd(text):
    """'2 Ty 700 Trieu' -> 2700 (trieu VND). Tra ve NaN neu khong parse duoc."""
    if pd.isna(text):
        return np.nan
    s = str(text).strip()
    if s == "" or s.lower() in {"nan", "thoa thuan", "giá thỏa thuận"}:
        return np.nan
    ty_match = re.search(r"([\d.,]+)\s*T[yỷ]", s, flags=re.IGNORECASE)
    trieu_match = re.search(r"([\d.,]+)\s*Tr[i1]ệu|Trieu|Triệu", s, flags=re.IGNORECASE)
    trieu_num_match = re.search(r"([\d.,]+)\s*Tri[eệ]u", s, flags=re.IGNORECASE)

    ty_val = 0.0
    trieu_val = 0.0
    if ty_match:
        ty_val = float(ty_match.group(1).replace(",", "."))
    if trieu_num_match:
        trieu_val = float(trieu_num_match.group(1).replace(",", "."))

    if ty_match or trieu_num_match:
        return ty_val * 1000 + trieu_val

    # fallback: chuoi chi co so (vd '450' hieu la 450 trieu), hoac so co dau cham/phay
    only_num = re.sub(r"[^\d.,]", "", s)
    if only_num == "":
        return np.nan
    try:
        return float(only_num.replace(",", ""))
    except ValueError:
        return np.nan


def split_brand_model(car_name):
    """Tach 'Mazda 3 1.5L Luxury' -> brand='Mazda', model='3 1.5L Luxury'."""
    if pd.isna(car_name):
        return np.nan, np.nan
    parts = str(car_name).strip().split(maxsplit=1)
    brand = parts[0] if len(parts) >= 1 else np.nan
    model = parts[1] if len(parts) >= 2 else np.nan
    return brand, model


def load_raw(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    return df


def clean_dataframe(df, min_price=30, max_price=60000, max_kms=1_000_000,
                     min_year=1980, max_year=2027):
    """
    Lam sach theo dung mo ta trong README:
    - parse gia ve trieu VND
    - bo dong thieu, bo trung theo url
    - bo dong nam khong hop le (loi crawl / lech cot)
    - tach brand/model tu car_name
    - loc ngoai lai: km, so cua/ghe, gia
    Tra ve DataFrame da lam sach voi FEATURE_COLUMNS + TARGET_COLUMN.
    """
    d = df.copy()
    d.columns = [c.strip() for c in d.columns]
    d = d.rename(columns={"driven kms": "driven_kms"})

    # 1) parse gia
    d["price"] = d["price"].apply(parse_price_to_million_vnd)

    # 2) ep kieu so, loai dong loi crawl (vd nam khong phai so)
    d["year"] = pd.to_numeric(d["year"], errors="coerce")
    d["driven_kms"] = pd.to_numeric(d["driven_kms"], errors="coerce")
    d["num_of_door"] = pd.to_numeric(d["num_of_door"], errors="coerce")
    d["num_of_seat"] = pd.to_numeric(d["num_of_seat"], errors="coerce")

    # 3) tach brand / model
    brand_model = d["car_name"].apply(split_brand_model)
    d["brand"] = brand_model.apply(lambda t: t[0])
    d["model"] = brand_model.apply(lambda t: t[1])

    # 4) bo dong thieu truong quan trong
    required = ["price", "year", "driven_kms", "brand", "series",
                "assemble_place", "engine_type", "transmission",
                "num_of_door", "num_of_seat", "url"]
    d = d.dropna(subset=required)

    # 5) bo trung theo url
    d = d.drop_duplicates(subset=["url"])

    # 6) loc ngoai lai
    d = d[(d["year"] >= min_year) & (d["year"] <= max_year)]
    d = d[(d["driven_kms"] >= 0) & (d["driven_kms"] <= max_kms)]
    d = d[(d["num_of_door"] >= 2) & (d["num_of_door"] <= 6)]
    d = d[(d["num_of_seat"] >= 2) & (d["num_of_seat"] <= 16)]
    d = d[(d["price"] >= min_price) & (d["price"] <= max_price)]

    d["year"] = d["year"].astype(int)
    d["num_of_door"] = d["num_of_door"].astype(int)
    d["num_of_seat"] = d["num_of_seat"].astype(int)

    out = d[FEATURE_COLUMNS + [TARGET_COLUMN]].reset_index(drop=True)
    return out


def load_clean(csv_path, **kwargs):
    return clean_dataframe(load_raw(csv_path), **kwargs)
